In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: one fixed raw-gradient 1:1 sum step

Run all evaluates exactly one adopted candidate, or resumes the same Drive
task. It reads `base`, `units/aggregate_payload.pt` and
`units/state_G16.pt` from the completed BASE-Alignment run. The BASE clamped
float RGB is reused from its already decoded source path. These paths and the
editable source reference are ordinary locators; there is no source-identity
or exact dependency-version admission gate.

The fixed construction preserves each saved gradient in its original scalar
units: `d=(g_A+g_G)/||g_A+g_G||`, then `SUM_STEP=BASE+0.10*d`. The weights
are exactly 1:1. There is no separate gradient normalization, endpoint
projection, scan, second step, rollback, alternate direction, new VJP or
replacement BASE decode. The support mask and original BASE radius cap 1
remain. A missing input, invalid/nonfinite tensor or zero gradient sum is an
explicit saved engineering failure.

BASE q, all32 margins and all22 correlations are reused from the saved
gradient unit. Only SUM_STEP is decoded and read. Both views report every bit
delta, every state branch, actual max-wrong offset/G, fixed G16, fixed-roster
mean A, positive/nonpositive counts, and distances from mu=.02 and gamma=.01.
Mu/gamma are descriptive only. Lost bits, a wrong-state branch change, G
decrease or poor quality are retained complete observations.

Quality uses decoder-output clamped float RGB in [0,1]. Aggregate, every
frame, adjacent temporal residual, actual peak pixels and their neighborhoods
are saved. Numeric residuals use the full post-clamp difference range;
[-0.25,0.25] applies only to display.

Nominal uninterrupted new GPU work is **1 VAE load + 1 ordinary decode
(46 chunks)**. VJP, gradient decode, encode, DiT, native/scheduler and codec
work are zero. Candidate latent commits before model loading, ordinary RGB
commits before readout, and scoring commits immediately. Every 60 seconds the
Journal records durable progress. Repeated Run all skips completed units and
a complete run loads no model. An interrupted unfinished decode may repeat;
physical calls, chunks, attempts and failures remain cumulative.

Stable task: MyDrive/Video-WM/Local-Joint-State-Payload-V1-Gradient-Sum/
resumable-gradient-sum-v1/. This one source point and one finite step do not
add an independent source video or independent sample. They do not establish
blind recovery, codec survival, FPR, independent-sample behavior,
generalization or scientific PASS. GPU/Colab/Drive execution is user-run.
Return the entire stable task directory.

In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable GitHub branch/tag/ref.
FIXED_CONFIG = {'schema': 'local-joint-gradient-sum-v1', 'source_run': 'resumable-base-alignment-v1/run', 'source_arm': 'JOINT_BASE', 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'revision': '0fad780a534b6463e45facd96134c9f345acfa5b'}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5, 'cap': 1.0}, 'gradient_weights': {'aggregate_payload': 1, 'state_G16': 1}, 'step': 0.1, 'device': 'cuda', 'inputs': {'alignment_run': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Base-Alignment/resumable-base-alignment-v1/run', 'base_float_rgb': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1/20261009T132316545817Z/run/joint/float_rgb.pt'}}
EXPECTED_COST = {'vae_load': 1, 'decode': 1, 'decoder_vjp': 0, 'gradient_decode': 0, 'ordinary_decode': 1, 'ordinary_chunks': 46, 'encode': 0, 'dit': 0, 'native': 0, 'scheduler': 0, 'codec': 0, 'new_windows': 88, 'new_chips': 1408, 'new_metrics': 55, 'reused_windows': 88, 'reused_chips': 1408, 'reused_metrics': 55, 'reported_windows': 176, 'reported_chips': 2816, 'reported_metrics': 110, 'reused_gradient_count': 2, 'new_view_count': 1, 'reused_view_count': 1, 'source_points': 1, 'new_source_endpoints': 1, 'independent_source_videos': 0, 'independent_samples': 0}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Gradient-Sum')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / 'resumable-gradient-sum-v1'  # Same logical task on every Run all.
OUTPUT.mkdir(exist_ok=True)
ATTEMPT_OUTPUT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT_OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-Gradient-Sum-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(ATTEMPT_OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (ATTEMPT_OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_gradient_sum_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))
        actual = json.loads((RUN_OUTPUT / 'result.json').read_text())
        fixed_slots.update(run_status=actual.get('status'),
            views=actual.get('views', fixed_slots['views']), counts=actual.get('counts'),
            ordinary_chunks=actual.get('ordinary_chunks'))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    views={name: dict(status='NOT_RUN', expected_windows=88, expected_chips=1408, expected_metrics=55)
           for name in ('BASE', 'SUM_STEP')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
write_json(ATTEMPT_OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Fixed GRADIENT-SUM output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_gradient_sum_v1_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    # A repeated Run all may retain packages from an earlier extracted workspace.
    import importlib
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(ATTEMPT_OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
from runtime.wan.local_joint_gradient_sum_v1 import model_work_remaining
try:
    NEEDS_MODEL = model_work_remaining(RUN_OUTPUT)
    probe_path = OUTPUT / 'dependency_probe.json'
    probe_lines = [
        "import json, pathlib, sys, torch, numpy, PIL",
        "row = dict(python=sys.version, torch=str(torch.__version__), model_work_remaining=" + repr(NEEDS_MODEL) + ")",
    ]
    if NEEDS_MODEL:
        probe_lines += ["from diffusers import AutoencoderKLWan"]
        if str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
            probe_lines += ["assert torch.cuda.is_available(), 'CUDA is needed for the unfinished ordinary decode'",
                            "row.update(cuda_available=True, device=torch.cuda.get_device_name(0))"]
    probe_lines += ["pathlib.Path(" + repr(str(probe_path)) + ").write_text(json.dumps(row))"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        packages = ['torch', 'numpy', 'pillow']
        if NEEDS_MODEL:
            packages += ['diffusers', 'transformers', 'accelerate', 'safetensors',
                         'huggingface-hub', 'sentencepiece', 'ftfy']
        logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    environment_receipt = json.loads(probe_path.read_text())
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode,
        source_and_versions_are_location_notes=True, checkpoint_interval_seconds=60)
    write_json(ATTEMPT_OUTPUT / 'environment_receipt.json', environment_receipt)
    print('Remaining model work:', NEEDS_MODEL, '; task:', OUTPUT, flush=True)
except BaseException as exc:
    record_failure('ENVIRONMENT_SETUP', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_gradient_sum_v1_run',
               '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_GRADIENT_SUM', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_gradient_sum_v1 import finalize_interrupted
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.exists() else None
    write_json(ATTEMPT_OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] != 'COMPLETE':
        raise RuntimeError('GRADIENT-SUM engineering interruption; retain entire output: ' + str(OUTPUT))
    fixed_slots.update(status=result['status'], result='run/result.json',
        views=result['views'], counts=result['counts'], ordinary_chunks=result['ordinary_chunks'])
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_GRADIENT_SUM', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
comparison = json.loads((RUN_OUTPUT / 'comparison.json').read_text())
print('Engineering status:', result['status'])
print('Nominal uninterrupted budget:', result['expected_cost'])
print('Cumulative physical calls:', result['counts'])
print('This Run all:', result.get('current_attempt_counts', {}))
print('Ordinary chunks:', result['ordinary_chunks'])
print('Fixed construction:', result['steps']['construct'])
for name, row in result['views'].items():
    print(name, row['status'], 'G=', row.get('state_gap'),
          'G16=', row.get('fixed_G16'), 'meanA=', row.get('fixed_mean_A'),
          'positive/nonpositive=', row.get('positive_count'), row.get('nonpositive_count'),
          'active wrong=', row.get('actual_max_wrong_offsets'), 'reason=', row.get('reason'))
print('All 32 bit deltas:', comparison['deltas']['margins'])
print('All 22 state deltas:', comparison['deltas']['correlations'])
print('Descriptive mu/gamma distances:', comparison['descriptive_distances'])
print('Quality versus BASE:', {name: None if row is None else {
    'rmse': row['rmse'], 'max_abs': row['max_abs'],
    'temporal_residual_rmse': row['temporal_residual_rmse'],
    'temporal_residual_max_abs': row['temporal_residual_max_abs'],
    'peak_events': row['peak_events']} for name, row in comparison['quality_vs_BASE'].items()})
print(result['ceiling'])
print('Return the entire output folder, including failures:', OUTPUT)